# 統合データ品質監査（2026-10-20）

## tl;dr
正本DB version 8をSHA256と照合し、データ・時刻・ID・同着精算・特徴量をCLIで監査した。修正と残る制約は `DEEP_RESEARCH_RESULTS.md` に記録する。このノートブックは環境のカーネル通信権限制約により未実行。保存済み監査結果を表示し、実データが手元にある場合に再監査できる補助資料。2025年は既に確認済みの回顧評価であり、将来成績ではない。

## Context & Methods

### Key Assumptions
粒度は1レース×馬番。学習2023、選択2024、回顧2025。公表時刻不明のデータは2日・7日遅延仮定で扱う。原本PDFの作成時刻を公表時刻にしない。SQLと検査ロジックは `keiba_v48/integrated_data_audit.py` に保存。

原本：`PROJECT_STATE.json` のLibrary IDと版・SHA256。JRA公式ルール： https://www.jra.go.jp/kouza/baken/index.html 。評価方法の背景：Gneiting & Raftery (2007), DOI 10.1198/016214506000001437；Cawley & Talbot (2010), https://www.jmlr.org/papers/v11/cawley10a.html 。

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PROJECT_STATE.json').exists())
sys.path.insert(0, str(root / 'keiba_v48'))
saved = root / 'keiba_v48/reports/deep_audit/data_audit_lag2.json'
audit = json.loads(saved.read_text())
assert audit['input_db_sha256'] == json.loads((root/'PROJECT_STATE.json').read_text())['database']['sha256']
if os.environ.get('KEIBA_AUDIT_DB'):
    from integrated_data_audit import audit_database
    audit = audit_database(os.environ['KEIBA_AUDIT_DB'], with_features=True)
print('Input SHA256:', audit['input_db_sha256'])
print('Read only / unchanged:', audit['read_only'], audit['database_unchanged'])

## Data

原本行・有効な全頭フィールド・除外を分ける。未連携の出走行は仮の公式IDにせず、他レースの履歴と結合しない。

In [ ]:
display(pd.DataFrame([audit['usable']]))
coverage = pd.DataFrame(audit['identity_by_year'])
display(coverage[['year','rows','horse_missing','horse_missing_fraction','jockey_missing']].round(4))
print('Most recent race:', audit['race_date_range']['max'])
print('Age as of audit date:', audit['source_result_age_days'], 'days')

## Results

同着で「全馬が3着以内」という判定だけでは、正しいワイド・3連複の的中条件を満たさない。同着は順位に割り当てられる枠数で判定する。以下の数は実際の購入点数ではなく、旧式判定で誤判定し得る組合せの数。

In [ ]:
d = audit['dead_heat_audit']
display(pd.DataFrame([{k:d[k] for k in ('races','old_rank_le3_false_hit_combinations','corrected_false_hits')}]))
print('Grain duplicates:', audit['runner_key_duplicate_groups'])
print('Identity disagreements:', audit['identity_link_disagreements'])
print('Incomplete accepted fields:', audit['full_field_headcount_mismatches'])
print('Unlinked keys reused:', audit['unlinked_isolation']['cross_race_key_reuse'])
if 'class_header_repair' in audit:
    display(pd.DataFrame([audit['class_header_repair']['summary']]))

## Takeaways

同着精算、頭数検査、仮想資金の集計、モデル昇格の承認条件、候補の永続保存を回帰テストで保護する。ID欠損・2026年履歴不足・当時の公開時刻は別の未解決課題。数値が改善しても、将来の固定予測100レースと収益基準を満たすまで稼働昇格しない。

全監査の再実行：`OPENBLAS_NUM_THREADS=1 python keiba_v48/integrated_data_audit.py --db /path/to/jra_2023_2025.sqlite --output audit.json --features`。原本を書き換えない。